In [ ]:
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
sys.path.append("..")
from src.data_generation.SSVI import sample_params, ssvi
from src.data_generation.data_preperation import sample_sparse_points
from src.evaluation.surface_eval import check_arbitrage

cfg = yaml.safe_load(open("../config.yaml"))

In [ ]:
np.random.seed(0)
ttms = np.geomspace(cfg["ttm"]["min"], cfg["ttm"]["max"], cfg["ttm"]["n_points"])

ks = np.linspace(cfg["k"]["min"], cfg["k"]["max"], cfg["k"]["n_points"])

rho, eta, gamma, v_bar, v0, kappa = sample_params(cfg, n=1000)

surfaces = ssvi(ttms, ks, rho, eta, gamma, v_bar, v0, kappa) 

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), subplot_kw={"projection": "3d"})
K, T = np.meshgrid(ks, ttms)

for i, ax in enumerate(axes.flat):
    iv = surfaces[i]
    ax.plot_surface(K, T, iv, cmap="viridis", edgecolor="none", alpha=0.9)
    ax.set_xlabel("k")
    ax.set_ylabel("tau")
    ax.set_zlabel("sigma")
    label = (
        f"rho={rho[i]:.2f}  eta={eta[i]:.2f}  gamma={gamma[i]:.2f} \n "
        f"vbar={v_bar[i]:.2f}  v0={v0[i]:.2f}  kappa={kappa[i]:.2f}"
    )
    ax.text2D(0.5, -0.05, label, transform=ax.transAxes, ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
cal_violations, butterfly_violations = check_arbitrage(surfaces, ttms, ks)
print(f"Calendar spread violations: {cal_violations.mean()*100}% ({cal_violations.sum()}/{len(surfaces)})")
print(f"Butterfly violations:       {butterfly_violations.mean()*100}% ({butterfly_violations.sum()}/{len(surfaces)})")
print(f"Any arbitrage:              {(cal_violations | butterfly_violations).mean()*100}%")

In [ ]:
n_diag = 100000
k_idx_d, t_idx_d = sample_sparse_points(ks, ttms, n_points=20, n_samples=n_diag)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
k_bins = cfg["k"]["n_points"]

axes[0].hist(ks[k_idx_d.ravel()], bins=k_bins, density=True, label="empirical")
k_w = np.exp(-0.5 * (ks / 0.25) ** 2); k_w /= k_w.sum() * (ks[1]-ks[0])
axes[0].plot(ks, k_w, "r-", label="target weight")
axes[0].set_title("k distribution"); axes[0].legend()

t_bins = np.append(ttms, ttms[-1] * (ttms[-1] / ttms[-2]))
axes[1].hist(ttms[t_idx_d.ravel()], bins=t_bins, density=True)
axes[1].set_xlabel(r"$\tau$"); axes[1].set_ylabel("density")
axes[1].set_title(r"$\tau$ distribution")

h = axes[2].hist2d(ks[k_idx_d.ravel()], ttms[t_idx_d.ravel()], bins=[k_bins, t_bins], density=True, cmap="plasma")
fig.colorbar(h[3], ax=axes[2], label="density")
axes[2].set_xlabel("k"); axes[2].set_ylabel(r"$\tau$"); axes[2].set_title("sampling density")

plt.tight_layout(); plt.show()

In [ ]:
from src.data_generation.noise import add_quote_noise

surf_i = 0
sigma = surfaces[surf_i]
TT, KK = np.meshgrid(ttms, ks, indexing="ij")
cfg["noise"]["jitter_sigma"] = 0.3

np.random.seed(0)
bid, ask = add_quote_noise(KK, TT, sigma, cfg["noise"], regime=2.0)

n_ttm = len(ttms)
n_cols = 3
n_rows = -(-n_ttm // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows), sharex=True, sharey=True)

for t_i, ax in enumerate(axes.ravel()):
    if t_i >= n_ttm:
        ax.axis("off")
        continue
    ax.plot(ks, sigma[t_i], "k-", label="SSVI slice")
    ax.errorbar(ks, (bid[t_i] + ask[t_i]) / 2, yerr=(ask[t_i] - bid[t_i]) / 2,
                fmt="C3.", capsize=2, ms=3, elinewidth=0.8, label="bid/ask quotes")
    ax.set_title(f"tau={ttms[t_i]:.3f}", fontsize=10)

for ax in axes[-1]:
    ax.set_xlabel("k")
for ax in axes[:, 0]:
    ax.set_ylabel("IV")
axes.ravel()[0].legend()
plt.tight_layout()
plt.show()